In [ ]:
import os
import yaml

import numpy as np
import torch
import timm

from ml_collections.config_dict import ConfigDict
from model.utils import load_model, load_checkpoint, load_dataset, evaluate

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [ ]:
step = 100

In [ ]:
config_file = yaml.load(open("../configs/step1_basic_contrastive.yaml","r"), yaml.FullLoader)

config = ConfigDict(config_file)

config.outdir = "../outputs/basic_contrastive_518556"

In [ ]:
pretrained_ckpt_dir = config.model.pretrained_image_encoder_ckpt_path
os.makedirs(pretrained_ckpt_dir, exist_ok=True)
pretrained_image_encoder = timm.create_model(
    "vit_large_patch16_224",
    img_size=224,
    patch_size=16,
    init_values=1e-5,
    num_classes=0,
    dynamic_img_size=True,
)
pretrained_image_encoder.load_state_dict(
    torch.load(
        os.path.join(pretrained_ckpt_dir, "pytorch_model.bin"), map_location="cpu"
    ),
    strict=True,
)
pretrained_image_encoder.eval()
    
model, optimizer, lr_scheduler = load_model(config, pretrained_image_encoder)

In [ ]:
load_checkpoint(model, optimizer, lr_scheduler, config.outdir+f"/checkpoint-{step}.pth")

In [ ]:
dict_embedding = torch.load(config.outdir+f"/embedding_step_{step-1}.pt")

In [ ]:
def plot_PCA(img_emb, spa_emb):
    # Concatenate embeddings
    all_embeddings = torch.cat([img_emb, spa_emb], dim=0)  # [256, 128]

    # Apply PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(all_embeddings.cpu().numpy())  # [256, 2]

    # Split for plotting
    pca_img = pca_result[:img_emb.size(0)]
    pca_spa = pca_result[img_emb.size(0):]

    # Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(pca_img[:, 0], pca_img[:, 1], label='Image', alpha=0.7, color='blue')
    plt.scatter(pca_spa[:, 0], pca_spa[:, 1], label='Spatial', alpha=0.7, color='orange')
    plt.legend()
    plt.title(f'PCA Projection (Step {step})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


    plt.figure(figsize=(6, 6))
    plt.scatter(pca_img[:, 0], pca_img[:, 1], label='Image', alpha=0.7, color='blue')
    plt.legend()
    plt.title(f'PCA Projection (Step {step})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


    plt.figure(figsize=(6, 6))
    plt.scatter(pca_spa[:, 0], pca_spa[:, 1], label='Spatial', alpha=0.7, color='orange')
    plt.legend()
    plt.title(f'PCA Projection (Step {step})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_cosine_sim(img_emb, spa_emb):
    img_emb = torch.nn.functional.normalize(img_emb, dim=-1)
    spa_emb = torch.nn.functional.normalize(spa_emb, dim=-1)

    logits = torch.mm(img_emb, spa_emb.t()) / config.model.infoNCE_temperature

    plt.imshow(logits)
    plt.colorbar(label='Cosine Similarity')
    plt.title("Cosine Similarity Heatmap")
    plt.xlabel("image embeddings")
    plt.ylabel("spatial embeddings")
    plt.show()
    
    
def plot_sphere(img_emb, spa_emb):
    z_i, z_j = img_emb, spa_emb

    # L2 normalize
    z_i_norm = torch.nn.functional.normalize(z_i, dim=-1).cpu().numpy()
    z_j_norm = torch.nn.functional.normalize(z_j, dim=-1).cpu().numpy()

    # Check shape == (N, 3)
    assert z_i_norm.shape[1] > 3, "Embedding dim must be 3 for 3D sphere visualization"

    # Plot
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(z_i_norm[:, 0], z_i_norm[:, 1], z_i_norm[:, 2], c='blue', label='z_i (image)', alpha=0.7)
    ax.scatter(z_j_norm[:, 0], z_j_norm[:, 1], z_j_norm[:, 2], c='red', label='z_j (spatial)', alpha=0.7)

    # Optional: draw unit sphere wireframe
    import numpy as np
    u, v = np.mgrid[0:2*np.pi:30j, 0:np.pi:15j]
    x = np.cos(u) * np.sin(v)
    y = np.sin(u) * np.sin(v)
    z = np.cos(v)
    ax.plot_wireframe(x, y, z, color='gray', alpha=0.1)

    ax.set_title("z_i and z_j on Unit Sphere")
    ax.legend()
    plt.show()

----

In [ ]:
img_raw = dict_embedding['image_input']
spa_raw = dict_embedding['spatial_input']

print("img_emb mean/std:", img_raw.mean().item(), img_raw.std().item())
print("spatial_emb mean/std:", spa_raw.mean().item(), spa_raw.std().item())

In [ ]:
img_raw[0]

In [ ]:
img_emb = dict_embedding['image_embedding']   # [128, 128]
spa_emb = dict_embedding['spatial_embedding'] # [128, 128]

print("img_emb mean/std:", img_emb.mean().item(), img_emb.std().item())
print("spatial_emb mean/std:", spa_emb.mean().item(), spa_emb.std().item())

In [ ]:
plot_PCA(img_emb, spa_emb)

In [ ]:
plot_cosine_sim(img_emb, spa_emb)

In [ ]:
plot_sphere(img_emb, spa_emb)